In [11]:
import pandas as pd
import sqlite3

In [6]:
df1_path = r"C:\Users\tonyw\Downloads\simplemaps_uscities_basicv1.78\uscities.csv" # us
df1 = pd.read_csv(df1_path)

df2_path = r"C:\Users\tonyw\Downloads\simplemaps_canadacities_basicv1.8\canadacities.csv" # canada
df2 = pd.read_csv(df2_path)

column_mapping = {
    'province_id': 'state_id',
    'province_name': 'state_name',
    'postal': 'zip'
}

df2_mapped = df2.rename(columns=column_mapping)


In [9]:
# Check if the column names are the same, and if not, align them
if not df1.columns.equals(df2_mapped.columns):
    # Add missing columns from df1 to df2
    missing_columns = df1.columns.difference(df2_mapped.columns)
    for column in missing_columns:
        df2_mapped[column] = None

    # Add missing columns from df2 to df1
    missing_columns = df2_mapped.columns.difference(df1.columns)
    for column in missing_columns:
        df1[column] = None

# Append df2 to df1
df1 = pd.concat([df1, df2_mapped], ignore_index=True)

In [12]:
# Create a connection to the SQLite database
conn = sqlite3.connect('ipg.sqlite')

# Insert df1 into the 'coordinates' table
df1.to_sql('coordinates', conn, if_exists='replace', index=False)

# Commit the changes and close the connection
conn.commit()
conn.close()


In [13]:
import pandas as pd
import sqlite3

# Connect to the SQLite database
conn = sqlite3.connect('ipg.sqlite')

# Read data from SQL tables into Pandas DataFrame
shipments_df = pd.read_sql_query('SELECT * FROM shipments', conn)
coordinates_df = pd.read_sql_query('SELECT * FROM coordinates', conn)

# Convert ship_to_city and state columns to lowercase for case-insensitive comparison
shipments_df['ship_to_city'] = shipments_df['ship_to_city'].str.lower()
shipments_df['state'] = shipments_df['state'].str.lower()
coordinates_df['city_ascii'] = coordinates_df['city_ascii'].str.lower()
coordinates_df['state_id'] = coordinates_df['state_id'].str.lower()

# Merge the DataFrames using case-insensitive comparison
merged_df = pd.merge(shipments_df, coordinates_df, 
                     left_on=['ship_to_city', 'state'], 
                     right_on=['city_ascii', 'state_id'], 
                     how='left')

# Select desired columns
result_df = merged_df[['bl_number', 'ship_to_city', 'state', 'ship_to_customer', 'lat', 'lng']]

# Print the result DataFrame
print(result_df)

     bl_number ship_to_city state                  ship_to_customer      lat  \
0      WZ1A912       irvine    ca                    MARUCHAN, INC.  33.6772   
1      WZ2A753      gardena    ca                      NISSIN FOODS  33.8943   
2      WZ3A004        chino    ca  WORLD CLASS DISTRIBUTION - CHINO  33.9836   
3      WZ3A004        chino    ca  WORLD CLASS DISTRIBUTION - CHINO  33.9836   
4      WZ3A060        lacey    wa      WORLD CLASS DISTRIBUTION -WA  47.0462   
...        ...          ...   ...                               ...      ...   
8002   2068803  white bluff    tn                   INTERSTATE PKG.  36.1005   
8003   2068803  white bluff    tn                   INTERSTATE PKG.  36.1005   
8004   2068803  white bluff    tn                   INTERSTATE PKG.  36.1005   
8005   2068805   cincinnati    oh                          CPG OHIO  39.1413   
8006   2068805   cincinnati    oh                          CPG OHIO  39.1413   

           lng  
0    -117.7738  
1    